# BGL Sliding Time-Window Sequencer

Builds **per-window event sequences** from a parsed BGL annotated log using a sliding time window.

**Input** — `*_bgl_annotated.parquet` produced by `ParserBGL.ipynb`  
**Output** — `*_bgl_sequences.parquet` — flat table of log lines with a `window_id` column,
sorted by `(window_id, unix_ts)`, ready for graph or model ingestion.

Each sequence is the ordered time-series of Drain template firings inside one sliding window.  
A window is labelled **anomalous** if at least one of its log lines is anomalous.

Default parameters (adjustable in Step 2):
* **Window size** — 60 min
* **Step size** — 30 min (50 % overlap between consecutive windows)

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, os.path.abspath(".."))  # project root

DATASET = "bgl"  # or "HDFS"
PROCESSED_DIR = Path(f"../data/processed/{DATASET}/")

## Step 1 — Load annotated log

Auto-picks the newest `*_bgl_annotated.parquet` from `data/processed/`.  
Override `input_file` manually to pin a specific run.

In [ ]:
candidates = sorted(PROCESSED_DIR.glob(f"*_{DATASET}_annotated.parquet"))
if not candidates:
    raise FileNotFoundError(
        f"No *_{DATASET}_annotated.parquet found in {PROCESSED_DIR}.\n"
        f"Run Parser{DATASET.upper()}.ipynb first to generate the annotated log file."
    )

input_file = candidates[-1]   # newest by YYYYMMDD_HHMM prefix
# input_file = PROCESSED_DIR / "20260422_2333_bgl_annotated.parquet"  # pin a specific run

RUN_TAG = input_file.stem.replace(f"_{DATASET}_annotated", "")
print(f"Run tag    : {RUN_TAG}")
print(f"Input file : {input_file}")

df = pd.read_parquet(input_file, engine="fastparquet")
df["parameters"] = df["parameters"].apply(json.loads)

# Drop rows with missing unix_ts (unparseable lines) and sort globally by time.
df = df.dropna(subset=["unix_ts"]).sort_values("unix_ts").reset_index(drop=True)
df["unix_ts"] = df["unix_ts"].astype(np.int64)

print(f"\nRows loaded : {len(df):,}")
print(f"Columns     : {df.columns.tolist()}")
df.head(3)

## Step 2 — Window parameters

Two parameters control the sliding window:

| Parameter | Default | Description |
|-----------|---------|-------------|
| `WINDOW_MINUTES` | 60 | Width of each window in minutes |
| `STEP_MINUTES` | 30 | Stride between consecutive window starts (T < W gives overlap; T == W → non-overlapping) |

Adjust both values and re-run Step 3 to change the windowing strategy.

In [ ]:
WINDOW_MINUTES = 5    # window width
STEP_MINUTES   = 1   # step size

WINDOW_SECONDS = WINDOW_MINUTES * 60
STEP_SECONDS   = STEP_MINUTES   * 60

t_min = int(df["unix_ts"].iloc[0])
t_max = int(df["unix_ts"].iloc[-1])
span_h = (t_max - t_min) / 3600
n_windows_est = max(0, int((t_max - t_min - WINDOW_SECONDS) / STEP_SECONDS) + 1)

print(f"Window size   : {WINDOW_MINUTES} min  ({WINDOW_SECONDS} s)")
print(f"Step size     : {STEP_MINUTES} min  ({STEP_SECONDS} s)")
print(f"Log span      : {span_h:.1f} h  ({t_min} → {t_max})")
print(f"Est. windows  : {n_windows_est:,}")

## Step 3 — Build sliding time windows

Assigns each log line to every window whose time interval contains it.  
A line at unix timestamp `t` belongs to window starting at `w` iff `w ≤ t < w + WINDOW_SECONDS`.

Consecutive `searchsorted` calls on the sorted `unix_ts` array make the assignment
O(W_count × log N) — fast even for millions of log lines.

`window_id` is the integer UNIX start timestamp of each window (seconds since epoch).  
Rows are duplicated across overlapping windows so the output is groupable by `window_id`,
matching the flat-parquet convention of the HDFS sequencer.

In [ ]:
import time

t0 = time.time()

unix_arr      = df["unix_ts"].values   # 1-D sorted int64 array
window_starts = np.arange(t_min, t_max - WINDOW_SECONDS + 1, STEP_SECONDS, dtype=np.int64)

row_indices: list[np.ndarray] = []
win_ids:     list[np.ndarray] = []

for w_start in window_starts:
    lo = int(np.searchsorted(unix_arr, w_start,               side="left"))
    hi = int(np.searchsorted(unix_arr, w_start + WINDOW_SECONDS, side="left"))
    n  = hi - lo
    if n == 0:
        continue
    row_indices.append(np.arange(lo, hi, dtype=np.int64))
    win_ids.append(np.full(n, w_start, dtype=np.int64))

all_rows = np.concatenate(row_indices)
all_wids = np.concatenate(win_ids)

df_windows = df.iloc[all_rows].copy()
df_windows["window_id"] = all_wids
df_windows = df_windows.sort_values(["window_id", "unix_ts"]).reset_index(drop=True)

elapsed      = time.time() - t0
n_nonempty   = len(win_ids)
mean_len     = len(all_rows) / n_nonempty if n_nonempty else 0

print(f"Windows (total / non-empty) : {len(window_starts):,} / {n_nonempty:,}")
print(f"Total (line, window) pairs  : {len(df_windows):,}")
print(f"Mean events / window        : {mean_len:.1f}")
print(f"Built in                    : {elapsed:.2f} s")

## Step 3b — (Optional) Sequence length analysis & capping

BGL windows have a very high length variance (median ~46, max ~177 k).  
**For graph-based models (GAE)** this is not a concern — each window is compressed into a
template-transition graph whose size is bounded by the number of unique Drain templates
(~100–300), regardless of how many raw events are in the window.

**For sequential models** (DeepLog, LogBERT) a cap or sub-sampling is needed.

`MAX_SEQ_LEN = None` → no filtering (default, safe for graph pipelines).  
Set it to e.g. `5_000` to drop windows that exceed that length.

## Step 4 — Sequence statistics

In [ ]:
import matplotlib.pyplot as plt

seq_lengths = df_windows.groupby("window_id").size().rename("length")
win_labels  = df_windows.groupby("window_id")["is_anomaly"].any()

n_total_w   = len(win_labels)
n_anomalous = int(win_labels.sum())
n_normal    = n_total_w - n_anomalous

print("Window-level anomaly distribution:")
print(f"  Total windows   : {n_total_w:,}")
print(f"  Normal windows  : {n_normal:,}  ({n_normal / n_total_w:.2%})")
print(f"  Anomalous       : {n_anomalous:,}  ({n_anomalous / n_total_w:.2%})")

print("\nSequence length (events per window):")
print(seq_lengths.describe().round(2).to_string())

print("\nPercentiles:")
pcts = [50, 75, 90, 95, 99, 100]
vals = np.percentile(seq_lengths, pcts)
print("  " + "  ".join(f"p{p}={v:.0f}" for p, v in zip(pcts, vals)))

# ── Distribution plot ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(
    f"Sequence length distribution  [{RUN_TAG}]  W={WINDOW_MINUTES} min  step={STEP_MINUTES} min",
    fontsize=11,
)

for ax, log_y in zip(axes, [False, True]):
    ax.hist(seq_lengths, bins=60, color="steelblue", edgecolor="white", linewidth=0.4)
    ax.axvline(seq_lengths.median(), color="red",    linestyle="--", linewidth=1.2,
               label=f"median={seq_lengths.median():.0f}")
    ax.axvline(seq_lengths.mean(),   color="orange", linestyle=":",  linewidth=1.2,
               label=f"mean={seq_lengths.mean():.1f}")
    ax.set_xlabel("Events per window")
    ax.set_ylabel("Number of windows")
    ax.set_title("Linear y" if not log_y else "Log y")
    if log_y:
        ax.set_yscale("log")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5 — Inspect sample windows

In [ ]:
def display_window(wid: int) -> None:
    """Pretty-print the ordered event sequence for a single time window."""
    sub = df_windows.loc[df_windows["window_id"] == wid]
    if sub.empty:
        print(f"Window {wid} not found.")
        return
    from datetime import datetime, timezone
    start_dt = datetime.fromtimestamp(wid,             tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    end_dt   = datetime.fromtimestamp(wid + WINDOW_SECONDS, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    print("=" * 100)
    print(f"Window ID   : {wid}")
    print(f"Time range  : {start_dt} → {end_dt}")
    print(f"Events      : {len(sub)}  |  anomalous: {sub['is_anomaly'].sum()}")
    print("=" * 100)
    for i, row in sub.iterrows():
        marker = "!" if row["is_anomaly"] else " "
        print(f"  [{marker}] {row['unix_ts']}  node={row['node_id']:<25}  cid={row['cluster_id']:>3}  {row['template'][:60]}")
    print()


# Show the shortest, median, and longest non-empty windows
sorted_wids = seq_lengths.sort_values().index.tolist()
sample_wids = {
    "shortest" : sorted_wids[0],
    "median"   : sorted_wids[len(sorted_wids) // 2],
    "longest"  : sorted_wids[-1],
}

for label, wid in sample_wids.items():
    print(f"\n{'━'*40} {label.upper()} {'━'*40}")
    display_window(wid)

## Step 6 — Save sequences to Parquet

In [ ]:
BGL_PROCESSED = PROCESSED_DIR
BGL_PROCESSED.mkdir(parents=True, exist_ok=True)

SEQUENCES_PATH = BGL_PROCESSED / f"{RUN_TAG}_bgl_sequences_W{WINDOW_MINUTES}S{STEP_MINUTES}.parquet"

df_save = df_windows.copy()
df_save["parameters"] = df_save["parameters"].apply(json.dumps)

# Cast Arrow-backed string columns to plain numpy object (fastparquet requirement)
for col in df_save.select_dtypes(include=["object", "string"]).columns:
    df_save[col] = df_save[col].astype(object)

df_save.to_parquet(SEQUENCES_PATH, index=False, engine="fastparquet")

print(f"Saved {len(df_save):,} rows across {df_save['window_id'].nunique():,} windows")
print(f"  → {SEQUENCES_PATH}  ({SEQUENCES_PATH.stat().st_size / 1_048_576:.1f} MB)")

# ── Reload in a later session ────────────────────────────────────────────────
# df_windows = pd.read_parquet(SEQUENCES_PATH, engine="fastparquet")
# df_windows["parameters"] = df_windows["parameters"].apply(json.loads)
# sequences = {wid: grp.reset_index(drop=True) for wid, grp in df_windows.groupby("window_id")}